[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/01_leslie_baselines.ipynb)

# Direct Leslie baselines (no autoencoder)

The paper's latent computations are judged against direct CMGDB runs on the
underlying maps themselves: the exact two-dimensional restriction of the
10-D Leslie contraction (section 4.1), and the three-class overcompensatory
Leslie map in its absorbing box (section 4.2). This notebook computes both.

The paper's baseline runs are heavier than a notebook session wants --
(24, 27, 28) for the 2-D map, and (29, 33, 36) for the 3-D map, which needs
a machine with roughly 100 GB of memory -- so the defaults below use coarse
grids that finish in seconds to minutes and show the same qualitative
structure. Set the paper values (on hardware that can afford them) to
reproduce the reference computations.

In [ ]:
# Install CMGDB. Running locally with CMGDB already installed, skip this cell.
!pip install -q CMGDB

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
SUBDIV_2D = (14, 16, 18)   # quick preview; paper value (24, 27, 28)
SUBDIV_3D = (12, 15, 18)   # quick preview; paper value (29, 33, 36) ~97 min
SUBDIV_LIMIT = 10_000
# ===========================================================================

## The two-dimensional Leslie baseline

The planar Leslie map at the paper's parameters (theta = 23.5, 23.5 and
survival 0.7) on the box [0, 90] x [0, 70]. This is exactly the dynamics the
paper's 10-D contraction embeds and the latent model recovers -- the extra
ambient coordinates are pure contractions and do not feed back. No padding,
matching the published reference.

In [ ]:
import math


def leslie_2d(x):
    th1, th2 = 23.5, 23.5   # fecundities; survival 0.7 -- the paper's values
    return [(th1 * x[0] + th2 * x[1]) * math.exp(-0.1 * (x[0] + x[1])), 0.7 * x[0]]


lower_2d = [0.0, 0.0]
upper_2d = [90.0, 70.0]
print(f"2-D box {lower_2d} -> {upper_2d}")

In [ ]:
import time

import CMGDB


def box_map_2d(rect):
    return CMGDB.BoxMap(leslie_2d, rect, padding=False)


model_2d = CMGDB.Model(SUBDIV_2D[1], SUBDIV_2D[2], SUBDIV_2D[0], SUBDIV_LIMIT,
                       lower_2d, upper_2d, box_map_2d)
started = time.perf_counter()
morse_graph_2d, _ = CMGDB.ComputeConleyMorseGraph(model_2d)
print(f"{morse_graph_2d.num_vertices()} Morse sets "
      f"in {time.perf_counter() - started:.1f}s at {SUBDIV_2D}")

In [ ]:
CMGDB.PlotMorseGraph(morse_graph_2d)

In [ ]:
CMGDB.PlotMorseSets(morse_graph_2d, xlabel="$x_1$", ylabel="$x_2$");

At the paper's (24, 27, 28) resolution this map resolves an invariant
circle, a period-three orbit, and their connecting structure; the coarse
preview typically shows fewer, merged sets.

## The three-dimensional Leslie baseline

The three-class overcompensatory Leslie map at theta = (28.9, 29.8, 22.0)
with survival 0.7, on the absorbing box the paper uses -- forward-invariant,
so the decomposition inside it is the whole recurrent structure. The
published screen at (29, 33, 36) resolves six Morse sets with two minimal
attractors. The full init = 29 grid is essential for that bistability: a
coarser initial grid leaves the transient region under-resolved and a
spurious connection collapses the two attractors into a chain. At init 29
the run needs roughly 100 GB of memory (with the transition cache off, as
below), which is why the preview here stays coarse.

In [ ]:
import math


def leslie_3d(point):
    total = point[0] + point[1] + point[2]
    return [
        (28.9 * point[0] + 29.8 * point[1] + 22.0 * point[2]) * math.exp(-0.1 * total),
        0.7 * point[0],
        0.7 * point[1],
    ]


lower_3d = [0.0, 0.0, 0.0]
upper_3d = [110.0, 77.0, 54.0]


def box_map_3d(rect):
    return CMGDB.BoxMap(leslie_3d, rect, padding=False)


model_3d = CMGDB.Model(SUBDIV_3D[1], SUBDIV_3D[2], SUBDIV_3D[0], SUBDIV_LIMIT,
                       lower_3d, upper_3d, box_map_3d)
started = time.perf_counter()
# cache_transition_graph=False keeps the memory bounded: the SCC passes
# re-evaluate the (cheap, batched-style) map instead of holding a per-level
# edge cache, which at the published init 29 would need 250 GB+.
morse_graph_3d, _ = CMGDB.ComputeConleyMorseGraph(model_3d, cache_transition_graph=False)
print(f"{morse_graph_3d.num_vertices()} Morse sets "
      f"in {time.perf_counter() - started:.1f}s at {SUBDIV_3D}")

In [ ]:
CMGDB.PlotMorseGraph(morse_graph_3d)

In [ ]:
CMGDB.PlotMorseSets3D(morse_graph_3d, xlabel="$x_1$", ylabel="$x_2$", zlabel="$x_3$");

The sets are drawn straight from the live Morse graph by CMGDB's cubical
surface renderer: only exposed faces are emitted, and orientation lighting
carries the depth. Tiny sets (a fixed point is a handful of cells) can be
inflated for display with `scale_factor` without changing what was
computed.